| [![home](../../media/navigation/home.png)](../../index.html) | [![next](../../media/navigation/next.png)](../../exercises/chapter-03-02/ex2.html) |
| :---: | :---: |
| • | Ex 3.2.2: Count co-occurrences in nuclei |

# 3.2 CellPose

## 3.2.1 Segmentation & tracking of yeasts

#### Required data:

| **Folder** | Description | Location | License |
| --- | --- | --- | --- |
| YIT-Benchmark2/TestSet8/RawData | Brightfield 2D time-series (2D+t) of yeast cells | [DOI: 10.1098/rsif.2016.0705](https://yeast-image-toolkit.org/) | CC-BY-4.0 |

#### Goals:

Along this exercise, we will try to:
- Segment instances of yeast cells using Cellpose.
- Track them along time.

Globally, we will use Cellpose from:
- Its basic GUI
- Napari


<center>
<img src="../../media/chapter03_02/cp-track.png" alt="tracked yeasts" width="33%" >
<br>   
<small>
Fig. 1.1 &ndash; Tracked yeasts cells.
</small>
</center>

### 3.2.1.1 Using Cellpose GUI

#### A. Goals

- Cellpose comes with a very basic and minimal GUI (**G**raphical **U**ser **I**nterface) so we can quickly try a model without installing anything else. 
- We will try to use it to segment our cells and see the limitations that we will run into quickly.


#### B. Launch the GUI

- Start by opening a new conda terminal and activate the environment in which you installed Cellpose with `conda activate cp-env`.
- Simply type `cellpose` and press [Enter] to open Cellpose's window.

**Note:**

If your image is a 3D stack, the command becomes `cellpose --do_3D --Zstack`


<center>
<img src="../../media/chapter03_02/cp-window.png" alt="cellpose window" width="33%" >
<br>   
<small>
Fig. 1.2 &ndash; Cellpose's basic interface.
</small>
</center>

- If you open the folder indicated in the "Required data" section, you will find a ".tif" file per point of time.
- You can drag-and-drop an image in the viewer or pass by the "File" > "Load image" menu of the interface.

#### C. Tuning the settings

- Using the "View" section, you can adjust the contrast of the channels present in your image.
- The first slider is responsible for the whole image if you have a single channel or the first if you have several channels.
- Adjusting the contrasts doesn't have any effect on the segmentation, it's just for preview.
- You cannot choose the LUT used for each channel, you are limited to red, green and blue.

**Warning:**

This basic tool only handles images containing up to 3 channels.

- In the "Segmentation" section, click on the triangle next to "additional settings" to unfold extra options.
- The first thing to adjust is the median diameter of objects.
- Edit this value until the magenta circle in the lower-left corner of the interface is approximately the diameter of most cells.
- You will certainly have a value between 20 and 25.
- This value will be used to rescale the image so objects are 30 pixels in diameter when fed to the model.
- You don't need to edit more settings for this image, you can run the inference by clicking on "run CPSAM".
- After a few seconds of waiting, you should see a labeled version of the image in the viewer.
- In the "Drawing" section, you can use:
    - the "MASKS ON" checkbox to toggle the masks display
    - the "outlines on" checkbox to show the contour of cells instead of solid labels.

<div style="display: flex; justify-content: center; gap: 1rem; text-align: center;">

  <div style="width: 25%;">
    <img src="../../media/chapter03_02/basic-before.png" alt="before segmentation" width="100%">
    <br>
    <small>Fig. 1.3 &ndash; Yeasts before segmentation.</small>
  </div>

  <div style="width: 25%;">
    <img src="../../media/chapter03_02/basic-after.png" alt="labels after segmentation" width="100%">
    <br>
    <small>Fig. 1.4 &ndash; Yeasts with superimposed labels.</small>
  </div>

  <div style="width: 25%;">
    <img src="../../media/chapter03_02/basic-contours.png" alt="contours after segmentation" width="100%">
    <br>
    <small>Fig. 1.5 &ndash; Contours instead of labels.</small>
  </div>

</div>

- Once you are satisfied with what you see, you can save the result by going to "File" > "Save masks as PNG/tif".
- No window will pop-up to ask you where yo save the result. It is automaticcaly saved next to the original image with the "cp_masks" suffix.

#### D. Limitations

- This GUI doesn't handle time, so if you want to process more time-points, you have to repeat the procedure done before.
- You can do a few more frames if you want to experiment with other settings, but we will try another way instead.

### 3.2.1.2 Using Cellpose from Napari

#### A. Goals

- Napari is an image viewer written in Python, and we can use it to execute Python code through a plugins mecanism.
- Pretty much any task can be executed in Napari as long as a plugin exists for it.
- In this part, we will use 3 plugins:
    - `cellpose-napari` to call Cellpose from Napari.
    - `set-calibration` to provide the physical pixel size and axes of the image.
    - `trackastra` to track our cells over time.

#### B. Setup the workspace

**Warning:**

Before continuing, delete the previous segmentation result from the folder containing input images (the "_cp_masks").

- Start by opening a new conda terminal and activate the environment in which you installed Cellpose with `conda activate cp-env`.
- Simply type `napari` and press [Enter] to open Cellpose's window.


<center>
<img src="../../media/chapter03_02/napari-window.png" alt="napari window" width="33%" >
<br>   
<small>
Fig. 1.2 &ndash; Napari window.
</small>
</center>

- In the top-bar menu, go to the "Plugins" menu and open:
    - "Calibration tool" > "Scale tool"
    - "cellpose-napari" > "Cellpose inference"
    - "Trackastra tracking"
- They should both show up as widgets in the right-side panel.
- In the top-bar menu, go to "File" > "Open folder..." and navigate to the folder containing the time series.
- If you are asked what to use to open it, go with "napari-builtins".
- The image should appear in the viewer.
- You should see a slider below to navigate through time.

#### C. Work on the time series

##### a. Calibration

- In the "Scale Tool" widget, you should see `Axes: '-3': 1 pixel (30) | '-2': 1 pixel (520) | '-1': 1 pixel (692)`.
- It means that the image we just loaded has three axes named '-3', '-2' and '-1' and they are 30, 520 and 692 units in size.
- However, Napari doesn't know what these axes are. 3 axes could mean:
    - A 3D image
    - A 2D time series
    - A 2D image with several channels
- We need to indicate what we actually have, as well as the physical size of pixels.
- Our pixels are 0.16 µm on the X and Y axes. We don't have a 3D stack, so you can let the 'Z' to 1.
- From the size of each axis, we can deduce that the first one is time (T) since we have 30 time points.
- In the "Axes" dropdown menu, you can choose `TYX` and then click on "Apply to all".
- If everything went well, you should see:
    - the scale bar in the lower-right corner of the viewer
    - the letter "T" on the left side of the slider below the viewer.

##### b. Cellpose

- In Cellpose's widget, set the "Main channel" dropdown to be our time series (if you have only one image in Napari, it should already be the case).
- Tune the "Median diameter" the same way as you did in Cellpose's GUI. A magenta circle should appear as well to indicate what it corresponds to.
- You can check the "Kill border?" box if you want to remove labels cut by the border.
- If you click on "Apply", in the very lower-right corner of Napari window:
    - a little animation should show up while Napari is working
    - if you click on "Activity", a loading bar will indicate what is going on.
- When the segmentation will be done, if you navigate through the time series, you will notice that the labels are not consistent (the colors blink). It means that cells don't have their proper identity over time.

##### c. Trackastra

- In Trackastra's widget, set:
    - the "Images" to be the original image 
    - the "Masks" to Cellpose's output
- Click on "Track".
- A new layer named "tracks" should appear to show you the track corresponding to each cell.
- A new layer named "masks_tracked" should appear on which the labels are now consistent across time.
- You can save this layer to reuse it wherever you need.
- To do so, click on the layer, go to "File" > "Save selected layers". It will produce a TIFF that you can reopen in Napari, Fiji, ...